# LG-CoTrain: Full Experiment Run with Optuna-Tuned Hyperparameters

This notebook runs the **complete experiment sweep** (all budgets x all seed sets x all events)
under each of the 6 stopping strategies, using **per-experiment Optuna-tuned hyperparameters**
instead of the fixed defaults used in [notebook 05](05_stopping_strategies_full_run.ipynb).

| Strategy | Key Idea |
|---|---|
| `baseline` | Original: stop when ensemble macro-F1 plateaus for `patience` epochs |
| `no_early_stopping` | Run all `finetune_max_epochs`; restore best-ever checkpoint (upper bound) |
| `per_class_patience` | Stop only when **every** class F1 has individually plateaued |
| `weighted_macro_f1` | Weight rare classes more in the stopping metric |
| `balanced_dev` | Resample dev set to equal class sizes for the stopping signal |
| `scaled_threshold` | Require a larger improvement delta for highly imbalanced events |

**Total experiments**: 6 strategies x 10 events x 4 budgets x 3 seeds = **720 runs**

**Key difference from notebook 05**: Each `(event, budget, seed_set)` combination uses its own
optimal `lr`, `batch_size`, `cotrain_epochs`, `finetune_patience`, `weight_decay`, and `warmup_ratio`
from per-experiment Optuna tuning.

Results are stored in `results/{PSEUDO_LABEL_SOURCE}/optuna-stop/{strategy}/`
(parallel to notebook 05's `results/.../stop/{strategy}/`).

## Resume Support

Every `(event, budget, seed_set)` combination is written to its own `metrics.json`
as soon as it completes. Re-running any cell automatically skips completed experiments.

In [2]:
import json
import statistics
import sys
import time
from pathlib import Path


def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(
        f"Cannot find repo root: no ancestor directory contains '{marker}/'. "
        "Run the notebook from inside the repository."
    )


repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np

from lg_cotrain.run_all import BUDGETS, SEED_SETS, format_summary_table, run_all_experiments
from lg_cotrain.parallel import run_experiments_parallel
from lg_cotrain.optuna_per_experiment import load_best_params


class ProgressTracker:
    """Track global progress across all strategies x events x budgets x seeds."""

    def __init__(self, total: int, already_done: int, start_time: float):
        self.total = total
        self.done = already_done
        self.start_time = start_time

    def update(self, event, budget, seed_set, status):
        self.done += 1
        elapsed = time.time() - self.start_time
        pct = 100.0 * self.done / self.total if self.total else 0
        elapsed_h = elapsed / 3600
        remaining = self.total - self.done
        eta_h = (elapsed / self.done) * remaining / 3600 if self.done > 0 else 0
        print(
            f"  [PROGRESS] {self.done}/{self.total} ({pct:.1f}%)"
            f"  |  Elapsed: {elapsed_h:.2f}h  |  ETA: {eta_h:.2f}h  |  {status}"
        )


print(f"Repo root : {repo_root}")
print(f"Budgets   : {BUDGETS}")
print(f"Seed sets : {SEED_SETS}")

Repo root : D:\Workspace\Co-Training
Budgets   : [5, 10, 25, 50]
Seed sets : [1, 2, 3]


In [3]:
# ---- Configuration ----

PSEUDO_LABEL_SOURCE = "gpt-4o"

# Number of GPUs for parallel execution (1 = sequential fallback)
NUM_GPUS = 2

# Optuna trial count to load (None = latest available)
N_TRIALS = 10

STRATEGIES = [
    "baseline",
    "no_early_stopping",
    "per_class_patience",
    "weighted_macro_f1",
    "balanced_dev",
    "scaled_threshold",
]

DATA_ROOT = str(repo_root / "data")

# Discover all events
TARGET_EVENTS = sorted(
    p.name for p in (Path(DATA_ROOT) / "original").iterdir() if p.is_dir()
)

# Each strategy gets its own results sub-folder under model/optuna-stop/
STRATEGY_RESULTS_ROOTS = {
    s: str(repo_root / "results" / PSEUDO_LABEL_SOURCE / "optuna-stop" / s)
    for s in STRATEGIES
}

# ---- Load Optuna best params for ALL (event, budget, seed_set) combos ----
OPTUNA_DIR = str(repo_root / "results" / "optuna" / "per_experiment")
optuna_results = load_best_params(
    storage_dir=OPTUNA_DIR,
    n_trials=N_TRIALS,
)

# Default hyperparams (fallback when Optuna results are missing)
DEFAULT_PARAMS = {
    "lr": 2e-5,
    "batch_size": 32,
    "cotrain_epochs": 10,
    "finetune_patience": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}

# Build per-experiment param lookup: (event, budget, seed_set) -> params dict
experiment_params = {}  # key -> 6-hyperparam dict
missing_keys = []
for event in TARGET_EVENTS:
    for budget in BUDGETS:
        for seed_set in SEED_SETS:
            key = (event, budget, seed_set)
            if key in optuna_results:
                experiment_params[key] = optuna_results[key]["best_params"]
            else:
                experiment_params[key] = None  # will use defaults
                missing_keys.append(key)

# ---- Print summary ----
expts_per_strategy = len(TARGET_EVENTS) * len(BUDGETS) * len(SEED_SETS)
total_runs = len(STRATEGIES) * expts_per_strategy
n_with_optuna = sum(1 for v in experiment_params.values() if v is not None)
n_with_default = sum(1 for v in experiment_params.values() if v is None)

print(f"Strategies    : {STRATEGIES}")
print(f"Events        : {len(TARGET_EVENTS)} events")
print(f"Per strategy  : {expts_per_strategy} experiments  ({len(TARGET_EVENTS)} events x {len(BUDGETS)} budgets x {len(SEED_SETS)} seeds)")
print(f"Grand total   : {total_runs} experiments  |  GPUs: {NUM_GPUS}")
print(f"Optuna params : {n_with_optuna}/{expts_per_strategy} loaded  |  {n_with_default} using defaults")
print(f"Optuna dir    : {OPTUNA_DIR}  (n_trials={N_TRIALS})")

if missing_keys:
    print(f"\nWARNING: {len(missing_keys)} experiment(s) missing Optuna results, using defaults:")
    for k in missing_keys[:10]:
        print(f"  {k}")
    if len(missing_keys) > 10:
        print(f"  ... and {len(missing_keys) - 10} more")

print()
for s, r in STRATEGY_RESULTS_ROOTS.items():
    print(f"  {s:<25} -> {r}")

Strategies    : ['baseline', 'no_early_stopping', 'per_class_patience', 'weighted_macro_f1', 'balanced_dev', 'scaled_threshold']
Events        : 10 events
Per strategy  : 120 experiments  (10 events x 4 budgets x 3 seeds)
Grand total   : 720 experiments  |  GPUs: 2
Optuna params : 120/120 loaded  |  0 using defaults
Optuna dir    : D:\Workspace\Co-Training\results\optuna\per_experiment  (n_trials=10)

  baseline                  -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\baseline
  no_early_stopping         -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\no_early_stopping
  per_class_patience        -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\per_class_patience
  weighted_macro_f1         -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\weighted_macro_f1
  balanced_dev              -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\balanced_dev
  scaled_threshold          -> D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\scaled_threshold


## Running Experiments

The loop below processes one strategy at a time, dispatching all events x budgets x seeds
in parallel across GPUs. Completed experiments are skipped automatically.

> **Runtime estimate**: With 2 GPUs, each strategy takes ~1.5-2.5 hours, ~9-15 hours total.
> Run overnight or split across sessions -- resume support ensures no work is lost.

In [ ]:
total_experiments = len(STRATEGIES) * expts_per_strategy

print(f"Total experiments : {total_experiments}")
print(f"Execution mode    : {'parallel (' + str(NUM_GPUS) + ' GPUs)' if NUM_GPUS > 1 else 'sequential'}")
print()

overall_start = time.time()
tracker = ProgressTracker(total_experiments, 0, overall_start)
all_strategy_results = {}  # strategy -> event -> list[result_dict]

for strategy in STRATEGIES:
    results_root = STRATEGY_RESULTS_ROOTS[strategy]
    strat_start = time.time()
    print(f"\n{'=' * 70}")
    print(f"Strategy: {strategy}  ->  {results_root}")
    print(f"{'=' * 70}")
    all_strategy_results[strategy] = {}

    # Build list of pending experiment configs, skipping completed ones
    pending_configs = []   # list of LGCoTrainConfig kwarg dicts
    pending_keys = []      # corresponding (event, budget, seed_set) tuples
    skipped_results = {}   # (event, budget, seed_set) -> result dict

    for event in TARGET_EVENTS:
        for budget in BUDGETS:
            for seed_set in SEED_SETS:
                metrics_path = (
                    Path(results_root) / event / f"{budget}_set{seed_set}" / "metrics.json"
                )

                if metrics_path.exists():
                    with open(metrics_path) as f:
                        skipped_results[(event, budget, seed_set)] = json.load(f)
                    tracker.update(event, budget, seed_set, "skipped")
                    continue

                # Per-experiment Optuna hyperparams (or defaults)
                p = experiment_params.get((event, budget, seed_set)) or DEFAULT_PARAMS
                pending_configs.append(dict(
                    event=event,
                    budget=budget,
                    seed_set=seed_set,
                    pseudo_label_source=PSEUDO_LABEL_SOURCE,
                    stopping_strategy=strategy,
                    lr=p["lr"],
                    batch_size=p["batch_size"],
                    cotrain_epochs=p["cotrain_epochs"],
                    finetune_patience=p["finetune_patience"],
                    weight_decay=p["weight_decay"],
                    warmup_ratio=p["warmup_ratio"],
                    data_root=DATA_ROOT,
                    results_root=results_root,
                ))
                pending_keys.append((event, budget, seed_set))

    n_skipped = len(skipped_results)
    n_pending = len(pending_configs)
    print(f"  Skipped (completed): {n_skipped}  |  Pending: {n_pending}")

    # Dispatch pending experiments
    parallel_results = {}  # (event, budget, seed_set) -> result dict

    if pending_configs and NUM_GPUS > 1:
        print(f"  Dispatching {n_pending} experiments across {NUM_GPUS} GPUs...")
        outcomes = run_experiments_parallel(
            pending_configs,
            num_gpus=NUM_GPUS,
            on_experiment_done=tracker.update,
        )
        for key, outcome in zip(pending_keys, outcomes):
            if outcome["status"] == "done" and outcome["result"] is not None:
                parallel_results[key] = outcome["result"]
            else:
                print(f"  WARNING: {key} failed")

    elif pending_configs:
        # Sequential fallback (NUM_GPUS == 1)
        for event in TARGET_EVENTS:
            # Gather per-event hyperparams for sequential path
            event_pending = [
                (k, c) for k, c in zip(pending_keys, pending_configs) if k[0] == event
            ]
            if not event_pending:
                continue

            # We must call run_all_experiments per unique param set, but since each
            # (event, budget, seed) has different params, run one at a time
            for key, cfg in event_pending:
                results = run_all_experiments(
                    key[0],
                    budgets=[key[1]],
                    seed_sets=[key[2]],
                    pseudo_label_source=PSEUDO_LABEL_SOURCE,
                    stopping_strategy=strategy,
                    lr=cfg["lr"],
                    batch_size=cfg["batch_size"],
                    cotrain_epochs=cfg["cotrain_epochs"],
                    finetune_patience=cfg["finetune_patience"],
                    weight_decay=cfg["weight_decay"],
                    warmup_ratio=cfg["warmup_ratio"],
                    data_root=DATA_ROOT,
                    results_root=results_root,
                    _on_experiment_done=tracker.update,
                )
                if results and results[0] is not None:
                    parallel_results[key] = results[0]

    # Merge all results into per-event lists
    all_results_flat = {**skipped_results, **parallel_results}
    for event in TARGET_EVENTS:
        event_results = [
            all_results_flat.get((event, b, s))
            for b in BUDGETS
            for s in SEED_SETS
        ]
        event_results = [r for r in event_results if r is not None]
        if event_results:
            all_strategy_results[strategy][event] = event_results
            # Print per-event summary
            print(f"\n  {format_summary_table(event_results, event)}")

    strat_elapsed = time.time() - strat_start
    print(f"\n  Strategy '{strategy}' done in {strat_elapsed / 3600:.2f}h ({strat_elapsed / 60:.1f}min)")

total_elapsed = time.time() - overall_start
print(f"\n{'=' * 70}")
print(f"All strategies complete in {total_elapsed / 3600:.2f}h ({total_elapsed / 60:.1f}min)")

Total experiments : 720
Execution mode    : parallel (2 GPUs)


Strategy: baseline  ->  D:\Workspace\Co-Training\results\gpt-4o\optuna-stop\baseline
  Skipped (completed): 0  |  Pending: 120
  Dispatching 120 experiments across 2 GPUs...


In [ ]:
# Load any results that already existed (re-run safe)
for strategy in STRATEGIES:
    results_root = Path(STRATEGY_RESULTS_ROOTS[strategy])
    if strategy not in all_strategy_results:
        all_strategy_results[strategy] = {}

    for event in TARGET_EVENTS:
        if event in all_strategy_results[strategy]:
            continue
        results = []
        for budget in BUDGETS:
            for seed_set in SEED_SETS:
                path = results_root / event / f"{budget}_set{seed_set}" / "metrics.json"
                if path.exists():
                    with open(path) as f:
                        results.append(json.load(f))
        if results:
            all_strategy_results[strategy][event] = results

# Build lookup: lookup[strategy][event][(budget, seed_set)] -> result
lookup = {}
for strategy in STRATEGIES:
    lookup[strategy] = {}
    for event in TARGET_EVENTS:
        lookup[strategy][event] = {}
        for r in all_strategy_results.get(strategy, {}).get(event, []):
            if r is not None:
                lookup[strategy][event][(r["budget"], r["seed_set"])] = r

# Coverage report
print("Experiments loaded per strategy:")
for strategy in STRATEGIES:
    n = sum(len(lookup[strategy][e]) for e in TARGET_EVENTS)
    expected = expts_per_strategy
    pct = 100 * n / expected if expected else 0
    print(f"  {strategy:<25}: {n:>3}/{expected}  ({pct:.0f}%)")

In [ ]:
# Summary tables: for each event, show mean macro-F1 per strategy x budget
# plus a delta-from-baseline table.

col_w = 10

for event in TARGET_EVENTS:
    print(f"\n{'=' * 70}")
    print(f"Event: {event}")
    print(f"{'=' * 70}")

    # --- Absolute macro-F1 ---
    print(f"{'Strategy':<26}" + "".join(f" B={b:<{col_w - 2}}" for b in BUDGETS) + " | Mean")
    print("-" * (26 + col_w * len(BUDGETS) + 7))

    baseline_means = {}
    for strategy in STRATEGIES:
        row = f"{strategy:<26}"
        budget_means = []
        for budget in BUDGETS:
            f1s = [
                lookup[strategy][event].get((budget, s), {}).get("test_macro_f1")
                for s in SEED_SETS
            ]
            f1s = [f for f in f1s if f is not None]
            if f1s:
                m = statistics.mean(f1s)
                sd = statistics.stdev(f1s) if len(f1s) >= 2 else 0.0
                budget_means.append(m)
                row += f" {m:.4f}+/-{sd:.4f}"
            else:
                budget_means.append(None)
                row += f" {'N/A':<{col_w}}"
        valid = [v for v in budget_means if v is not None]
        row += f" | {statistics.mean(valid):.4f}" if valid else " | N/A"
        print(row)
        if strategy == "baseline":
            baseline_means = dict(zip(BUDGETS, budget_means))

    # --- Delta vs baseline ---
    print()
    print(f"Delta vs baseline  (+) = better:")
    print(f"{'Strategy':<26}" + "".join(f" B={b:<{col_w - 2}}" for b in BUDGETS) + " | Mean delta")
    print("-" * (26 + col_w * len(BUDGETS) + 12))

    for strategy in STRATEGIES:
        if strategy == "baseline":
            continue
        row = f"{strategy:<26}"
        deltas = []
        for budget in BUDGETS:
            f1s = [
                lookup[strategy][event].get((budget, s), {}).get("test_macro_f1")
                for s in SEED_SETS
            ]
            f1s = [f for f in f1s if f is not None]
            base = baseline_means.get(budget)
            if f1s and base is not None:
                d = statistics.mean(f1s) - base
                deltas.append(d)
                sign = "+" if d >= 0 else ""
                row += f" {sign}{d:.4f}   "
            else:
                row += f" {'N/A':<{col_w}}"
        row += f" | {'+' if sum(deltas)/len(deltas)>=0 else ''}{sum(deltas)/len(deltas):.4f}" if deltas else " | N/A"
        print(row)

In [ ]:
# Grouped bar chart: macro-F1 by budget, grouped bars per strategy
# One subplot per event (mean +/- std across 3 seeds)

n_events     = len(TARGET_EVENTS)
n_strategies = len(STRATEGIES)
bar_width    = 0.8 / n_strategies
colors       = plt.cm.tab10(np.linspace(0, 1, n_strategies))

# Layout: up to 5 events per row
ncols = min(5, n_events)
nrows = (n_events + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows), sharey=False)
axes_flat = np.array(axes).flatten() if n_events > 1 else [axes]

for ax, event in zip(axes_flat, TARGET_EVENTS):
    x = np.arange(len(BUDGETS))
    for i, (strategy, color) in enumerate(zip(STRATEGIES, colors)):
        means, errs = [], []
        for budget in BUDGETS:
            f1s = [
                lookup[strategy][event].get((budget, s), {}).get("test_macro_f1")
                for s in SEED_SETS
            ]
            f1s = [f for f in f1s if f is not None]
            means.append(statistics.mean(f1s) if f1s else 0)
            errs.append(statistics.stdev(f1s) if len(f1s) >= 2 else 0)
        offset = (i - n_strategies / 2 + 0.5) * bar_width
        ax.bar(
            x + offset, means, bar_width * 0.9,
            yerr=errs, capsize=2,
            label=strategy, color=color, alpha=0.85,
        )
    ax.set_title(event.replace("_", " ").title(), fontsize=9)
    ax.set_xlabel("Budget")
    ax.set_ylabel("Macro-F1")
    ax.set_xticks(x)
    ax.set_xticklabels([str(b) for b in BUDGETS])
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)

# Hide unused subplots
for ax in axes_flat[n_events:]:
    ax.set_visible(False)

axes_flat[0].legend(fontsize=7, loc="upper left", framealpha=0.7)
fig.suptitle(
    f"Stopping Strategy Comparison (Optuna Hyperparams) -- All Budgets & Seeds\n"
    f"(pseudo-labels: {PSEUDO_LABEL_SOURCE})",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-event summary: for each strategy, show mean macro-F1 across all events and budgets.
# Useful for picking a single "best" strategy to recommend.

print("Grand summary -- mean macro-F1 across all events and seeds (Optuna hyperparams)\n")
print(f"{'Strategy':<26}" + "".join(f" B={b:<8}" for b in BUDGETS) + " | Overall")
print("-" * (26 + 10 * len(BUDGETS) + 10))

for strategy in STRATEGIES:
    row = f"{strategy:<26}"
    all_f1s = []
    for budget in BUDGETS:
        f1s = [
            lookup[strategy][e].get((budget, s), {}).get("test_macro_f1")
            for e in TARGET_EVENTS
            for s in SEED_SETS
        ]
        f1s = [f for f in f1s if f is not None]
        if f1s:
            m = statistics.mean(f1s)
            sd = statistics.stdev(f1s) if len(f1s) >= 2 else 0
            all_f1s.extend(f1s)
            row += f" {m:.4f}+/-{sd:.4f}"
        else:
            row += f" {'N/A':<10}"
    overall = statistics.mean(all_f1s) if all_f1s else None
    row += f" | {overall:.4f}" if overall is not None else " | N/A"
    print(row)

# Per-class F1 heatmap for each event at budget=5 (hardest case)
from lg_cotrain.data_loading import CLASS_LABELS

HEATMAP_BUDGET = 5
print(f"\nPer-class F1 heatmaps at budget={HEATMAP_BUDGET} (hardest imbalance scenario)")

for event in TARGET_EVENTS:
    strategies_with_data = []
    class_f1_matrix = []

    for strategy in STRATEGIES:
        per_class_all_seeds = [
            lookup[strategy][event][(HEATMAP_BUDGET, s)]["test_per_class_f1"]
            for s in SEED_SETS
            if (HEATMAP_BUDGET, s) in lookup[strategy][event]
            and "test_per_class_f1" in lookup[strategy][event][(HEATMAP_BUDGET, s)]
        ]
        if per_class_all_seeds:
            mean_per_class = [
                statistics.mean(seed[i] for seed in per_class_all_seeds)
                for i in range(len(per_class_all_seeds[0]))
            ]
            strategies_with_data.append(strategy)
            class_f1_matrix.append(mean_per_class)

    if not strategies_with_data:
        print(f"  No per-class data for {event} at budget={HEATMAP_BUDGET}, skipping.")
        continue

    data = np.array(class_f1_matrix)
    n_classes = data.shape[1]
    class_labels = CLASS_LABELS[:n_classes] if n_classes <= len(CLASS_LABELS) else [
        f"class_{i}" for i in range(n_classes)
    ]

    fig, ax = plt.subplots(
        figsize=(max(9, n_classes * 0.75), len(strategies_with_data) * 0.65 + 1.8)
    )
    im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(strategies_with_data)))
    ax.set_yticklabels(strategies_with_data, fontsize=9)
    ax.set_title(
        f"{event}  |  Budget={HEATMAP_BUDGET} (Optuna)  |  Per-class F1 (mean across seeds)",
        fontsize=10,
    )

    for i in range(len(strategies_with_data)):
        for j in range(n_classes):
            val = data[i, j]
            color = "black" if 0.25 < val < 0.75 else "white"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

    fig.colorbar(im, ax=ax, label="F1 Score")
    plt.tight_layout()
    plt.show()

## Comparison: Fixed Hyperparams (Notebook 05) vs Optuna-Tuned

Load the full-run results from notebook 05 (fixed defaults) and compare them
side-by-side with the Optuna-tuned results from this notebook.

In [ ]:
# Load notebook 05 results (fixed hyperparams) from results/gpt-4o/stop/
FIXED_RESULTS_ROOTS = {
    s: str(repo_root / "results" / PSEUDO_LABEL_SOURCE / "stop" / s)
    for s in STRATEGIES
}

fixed_lookup = {}  # strategy -> event -> {(budget, seed): result}
for strategy in STRATEGIES:
    fixed_lookup[strategy] = {}
    for event in TARGET_EVENTS:
        fixed_lookup[strategy][event] = {}
        for budget in BUDGETS:
            for seed_set in SEED_SETS:
                path = (
                    Path(FIXED_RESULTS_ROOTS[strategy])
                    / event / f"{budget}_set{seed_set}" / "metrics.json"
                )
                if path.exists():
                    with open(path) as f:
                        fixed_lookup[strategy][event][(budget, seed_set)] = json.load(f)

# ---- Side-by-side comparison for baseline strategy: per event x budget ----
strategy = "baseline"
print(f"Fixed vs Optuna: Mean Test Macro-F1 (strategy={strategy})")
print("=" * 90)

for event in TARGET_EVENTS:
    print(f"\nEvent: {event}")
    print(f"{'Budget':<8} {'Fixed':>12} {'Optuna':>12} {'Delta':>10}")
    print("-" * 45)

    for budget in BUDGETS:
        fixed_f1s = [
            fixed_lookup[strategy][event].get((budget, s), {}).get("test_macro_f1")
            for s in SEED_SETS
        ]
        optuna_f1s = [
            lookup[strategy][event].get((budget, s), {}).get("test_macro_f1")
            for s in SEED_SETS
        ]
        fixed_f1s = [f for f in fixed_f1s if f is not None]
        optuna_f1s = [f for f in optuna_f1s if f is not None]

        f_str = f"{statistics.mean(fixed_f1s):.4f}" if fixed_f1s else "N/A"
        o_str = f"{statistics.mean(optuna_f1s):.4f}" if optuna_f1s else "N/A"

        if fixed_f1s and optuna_f1s:
            d = statistics.mean(optuna_f1s) - statistics.mean(fixed_f1s)
            sign = "+" if d >= 0 else ""
            d_str = f"{sign}{d:.4f}"
        else:
            d_str = "N/A"

        print(f"{budget:<8} {f_str:>12} {o_str:>12} {d_str:>10}")

# ---- Grand summary: overall mean F1 fixed vs Optuna per budget ----
print(f"\n{'=' * 90}")
print(f"Grand Summary: Fixed vs Optuna (baseline strategy, all events)")
print(f"{'Budget':<8} {'Fixed':>12} {'Optuna':>12} {'Delta':>10}")
print("-" * 45)

all_fixed_vals = []
all_optuna_vals = []

for budget in BUDGETS:
    fixed_f1s = [
        fixed_lookup[strategy][e].get((budget, s), {}).get("test_macro_f1")
        for e in TARGET_EVENTS for s in SEED_SETS
    ]
    optuna_f1s = [
        lookup[strategy][e].get((budget, s), {}).get("test_macro_f1")
        for e in TARGET_EVENTS for s in SEED_SETS
    ]
    fixed_f1s = [f for f in fixed_f1s if f is not None]
    optuna_f1s = [f for f in optuna_f1s if f is not None]

    f_str = f"{statistics.mean(fixed_f1s):.4f}" if fixed_f1s else "N/A"
    o_str = f"{statistics.mean(optuna_f1s):.4f}" if optuna_f1s else "N/A"

    if fixed_f1s and optuna_f1s:
        d = statistics.mean(optuna_f1s) - statistics.mean(fixed_f1s)
        sign = "+" if d >= 0 else ""
        d_str = f"{sign}{d:.4f}"
        all_fixed_vals.extend(fixed_f1s)
        all_optuna_vals.extend(optuna_f1s)
    else:
        d_str = "N/A"

    print(f"{budget:<8} {f_str:>12} {o_str:>12} {d_str:>10}")

if all_fixed_vals and all_optuna_vals:
    d_overall = statistics.mean(all_optuna_vals) - statistics.mean(all_fixed_vals)
    sign = "+" if d_overall >= 0 else ""
    print(f"{'Overall':<8} {statistics.mean(all_fixed_vals):>12.4f} {statistics.mean(all_optuna_vals):>12.4f} {sign}{d_overall:>9.4f}")

# ---- Bar chart: Fixed vs Optuna per budget (baseline strategy) ----
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(BUDGETS))
w = 0.35

fixed_means = []
optuna_means = []
for budget in BUDGETS:
    ff = [fixed_lookup[strategy][e].get((budget, s), {}).get("test_macro_f1")
          for e in TARGET_EVENTS for s in SEED_SETS]
    of = [lookup[strategy][e].get((budget, s), {}).get("test_macro_f1")
          for e in TARGET_EVENTS for s in SEED_SETS]
    ff = [f for f in ff if f is not None]
    of = [f for f in of if f is not None]
    fixed_means.append(statistics.mean(ff) if ff else 0)
    optuna_means.append(statistics.mean(of) if of else 0)

ax.bar(x - w / 2, fixed_means,  w, label="Fixed hyperparams (NB 05)", color="steelblue", alpha=0.85)
ax.bar(x + w / 2, optuna_means, w, label="Optuna-tuned (NB 10)",     color="darkorange", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("Budget (labels per class)")
ax.set_ylabel("Mean Test Macro-F1")
ax.set_ylim(0, 1)
ax.set_title(
    f"Baseline Stopping: Fixed vs Optuna Hyperparams\n"
    f"Mean across all events and seeds (pseudo-labels: {PSEUDO_LABEL_SOURCE})",
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rebuild multi-tab dashboard with all strategy result sets.

from lg_cotrain.dashboard import discover_result_sets, generate_html_multi

TOP_RESULTS_ROOT = str(repo_root / "results")

result_sets = discover_result_sets(TOP_RESULTS_ROOT)
print(f"Discovered {len(result_sets)} model(s):")
for model, types in result_sets.items():
    for exp_type, experiments in types.items():
        for name, path in experiments:
            print(f"  {model}/{exp_type}/{name:<20} -> {path}")

html = generate_html_multi(result_sets, data_root=DATA_ROOT)
dashboard_path = Path(TOP_RESULTS_ROOT) / "dashboard.html"
dashboard_path.write_text(html, encoding="utf-8")
print(f"\nDashboard written to: {dashboard_path}")
print("Open in a browser to compare all strategies across all budgets and events.")